# CEPHA29 landmark model — train + export (Kaggle T4)

Trains a 29-landmark HRNet-W32 on CEPHA29 with **CephaloHRNet** (MIT) and exports a
web-compatible ONNX (FP32 + static QDQ INT8) for Cephalometry Studio.

**Kaggle setup**
1. Settings → Accelerator: **GPU T4 x2** (one is used).
2. Settings → Internet: **On** (required — the notebook downloads the Aariz/CEPHA29 dataset
   from Figshare (~1.95 GB) and clones the repos).
3. (Optional) Add Data → add a Kaggle dataset containing `train/Cephalograms` to skip the download.
4. **Save & Run All (Commit)** — a committed run persists `/kaggle/working` (including the
   exported ONNX + SSM JSON) into the version Output, so nothing is lost if the session ends.
   Resume from `last.pt` if a run is interrupted.

Dataset: Figshare **Aariz Cephalometric Dataset** (= CEPHA29), CC BY 4.0 — https://doi.org/10.6084/m9.figshare.27986417

Time: ~5–10 h at 768² with a pretrained backbone on a T4.

In [ ]:
!nvidia-smi
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# torch is preinstalled on Kaggle. Install the remaining deps.
!pip -q install timm albumentations opencv-python-headless pyyaml onnx onnxruntime pillow numpy scipy

In [ ]:
import os
REPO_URL = 'https://github.com/Muhammad-Nabeel-Sh/CephalometryStudio.git'  # for scripts/export-landmark-onnx.py
%cd /kaggle/working
!rm -rf CephaloHRNet CephalometryStudio
!git clone --depth 1 https://github.com/Cestovatels/CephaloHRNet.git
!git clone --depth 1 $REPO_URL CephalometryStudio
print('cloned')

In [ ]:
import os, hashlib, zipfile, requests

DATASET = None
# 1) a Kaggle input dataset, if one was added
for base, dirs, _ in os.walk('/kaggle/input'):
    if os.path.basename(base) == 'train' and 'Cephalograms' in dirs:
        DATASET = os.path.dirname(base)
        break

# 2) else download Aariz (CEPHA29) from Figshare — CC BY 4.0
if DATASET is None:
    URL = 'https://ndownloader.figshare.com/files/51041642'
    MD5 = 'e0bd645bca6759abdae4f199d841bda6'
    SIZE = 2098209792
    ZIP = '/kaggle/working/Aariz.zip'
    if not os.path.exists(ZIP) or os.path.getsize(ZIP) != SIZE:
        with requests.get(URL, stream=True, headers={'User-Agent': 'Mozilla/5.0'}) as r:
            r.raise_for_status()
            with open(ZIP, 'wb') as f:
                for chunk in r.iter_content(1 << 20):
                    f.write(chunk)
    h = hashlib.md5()
    with open(ZIP, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    assert h.hexdigest() == MD5, 'Aariz.zip md5 mismatch — delete and re-run to re-download'
    with zipfile.ZipFile(ZIP) as z:
        z.extractall('/kaggle/working/aariz')
    for base, dirs, _ in os.walk('/kaggle/working/aariz'):
        if os.path.basename(base) == 'train' and 'Cephalograms' in dirs:
            DATASET = os.path.dirname(base)
            break

print('DATASET =', DATASET)
assert DATASET, 'Could not locate the CEPHA29/Aariz dataset (need internet ON or a Kaggle input).'

In [ ]:
import os
# Inspect the annotation layout and choose annotator dirs (averages Junior+Senior if both exist).
base = os.path.join(DATASET, 'train', 'Annotations')
ann_names = []
lc = os.path.join(base, 'Cephalometric Landmarks')
if os.path.isdir(lc):
    ann_names = sorted(d for d in os.listdir(lc) if os.path.isdir(os.path.join(lc, d)))
elif os.path.isdir(base):
    ann_names = sorted(d for d in os.listdir(base) if os.path.isdir(os.path.join(base, d)))
print('annotator dirs:', ann_names)
print('images:', len(os.listdir(os.path.join(DATASET, 'train', 'Cephalograms'))))

j = [a for a in ann_names if 'Junior' in a]
s = [a for a in ann_names if 'Senior' in a]
ANN_NAMES = [j[0], s[0]] if (j and s) else list(ann_names)

In [ ]:
import subprocess, sys
# imgsz is square (768 matches the app). batch 4 + amp fits a T4 16GB.
# If OOM: --imgsz 512. Keep batch >= 4 for stable BatchNorm.
cmd = [sys.executable, 'train.py',
       '--data', DATASET,
       '--model', 'hrnet_w32',
       '--imgsz', '768',
       '--batch', '4',
       '--epochs', '200',
       '--loss', 'awing',
       '--amp',
       '--device', '0',
       '--pretrained',
       '--workers', '2',
       '--patience', '30',
       '--project', 'runs/train',
       '--name', 'cepha29_w32_768']
if ANN_NAMES:
    cmd += ['--annotators'] + ANN_NAMES
print(' '.join(cmd))
subprocess.run(cmd, cwd='/kaggle/working/CephaloHRNet', check=True)
print('training done')

In [ ]:
import glob, os, shutil
bests = sorted(glob.glob('/kaggle/working/CephaloHRNet/runs/train/*/best.pt'), key=os.path.getmtime)
assert bests, 'best.pt not found — check the training output above.'
BEST = bests[-1]
print('BEST =', BEST)

# A few real cephalograms for INT8 calibration + a verification overlay.
imgs = [f for f in sorted(glob.glob(os.path.join(DATASET, 'train', 'Cephalograms', '*')))
        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'))][:16]
assert imgs, 'no training images found for calibration'
os.makedirs('/kaggle/working/calib', exist_ok=True)
for f in imgs:
    shutil.copy(f, '/kaggle/working/calib/' + os.path.basename(f))
SAMPLE = os.path.join('/kaggle/working/calib', os.path.basename(imgs[0]))
print('calib images:', len(imgs), '| sample:', os.path.basename(SAMPLE))

In [ ]:
import subprocess, sys, os
# Export FP32 + web-compatible static QDQ INT8 ONNX.
cmd = [sys.executable, 'CephalometryStudio/scripts/export-landmark-onnx.py',
       '--repo', '/kaggle/working/CephaloHRNet',
       '--weights', BEST,
       '--model', 'hrnet_w32', '--num-landmarks', '29',
       '--input-size', '768',
       '--out', '/kaggle/working/landmarks-cepha29.onnx',
       '--quantize', '--calib-dir', '/kaggle/working/calib',
       '--verify', SAMPLE]
print(' '.join(cmd))
subprocess.run(cmd, cwd='/kaggle/working', check=True)
print('\nOutputs in /kaggle/working:')
for f in sorted(os.listdir('/kaggle/working')):
    if f.endswith('.onnx') or f.endswith('-verify.png'):
        print(' ', f, round(os.path.getsize('/kaggle/working/' + f) / 1e6, 1), 'MB')

In [ ]:
import os, subprocess, sys
# Build the CEPHA29 statistical shape-model (SSM) prior used by the browser
# refinement pass (src/lib/shapeModel.js): a PCA of Procrustes-aligned shapes.
ssm_ann = next((a for a in ANN_NAMES if 'Senior' in a), None) or (ANN_NAMES[0] if ANN_NAMES else None)
ann_dir = (os.path.join(DATASET, 'train', 'Annotations', 'Cephalometric Landmarks', ssm_ann)
           if ssm_ann else os.path.join(DATASET, 'train', 'Annotations'))
print('SSM annotations:', ann_dir)
if os.path.isdir(ann_dir):
    r = subprocess.run([sys.executable, 'CephalometryStudio/scripts/build-shape-model.py',
        '--set', 'cepha29', '--ann-dir', ann_dir, '--k', '10',
        '--out', '/kaggle/working/shapeModel.cepha29.json'],
        cwd='/kaggle/working', check=False)
    print('shape-model returncode:', r.returncode)
else:
    print('WARNING: annotations not found; skipping SSM prior (set refine.shape=false in the app).')

In [ ]:
import os, glob, shutil, hashlib
def _sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for c in iter(lambda: f.read(1 << 20), b''):
            h.update(c)
    return h.hexdigest()

out = '/kaggle/working/deliverables'
os.makedirs(out, exist_ok=True)
for name in ('landmarks-cepha29.onnx', 'landmarks-cepha29.int8.onnx',
             'landmarks-cepha29-verify.png', 'shapeModel.cepha29.json'):
    for p in glob.glob('/kaggle/working/' + name):
        shutil.copy(p, out)
print('=== DELIVERABLES - /kaggle/working/deliverables ===')
for f in sorted(os.listdir(out)):
    p = os.path.join(out, f)
    print(f'{os.path.getsize(p)/1e6:9.2f} MB  {f}\n            sha256 {_sha256(p)}')

## Next
1. **Inspect** `landmarks-cepha29-verify.png` — every dot is labelled `index:symbol`; confirm it lands on the right anatomy (validates the 29-landmark channel order).
2. Download `landmarks-cepha29.int8.onnx` and its printed SHA-256.
3. Upload it to a GitHub Release / Hugging Face (CORS-enabled).
4. Update `src/data/landmarkModelInfo.js` (`landmarkSet: "cepha29"`, `inputSize: 768`, `url`, `sha256`, `enabled: true`) and build the CEPHA29 shape model (`python scripts/build-shape-model.py --set cepha29 --ann-dir <Annotations/.../Senior Orthodontists>`).